# Tabula Rasa AI — Quickstart Notebook

Train, evaluate, and interact with a **specialist math model** from scratch.

This notebook uses the [Tabula Rasa](https://github.com/NousResearch/tabula-rasa) framework to:
1. Create a tiny transformer from scratch
2. Train it on **addition** problems using curriculum learning
3. Evaluate its accuracy broken down by digit length
4. Visualize the training curves
5. Chat interactively with the trained model

**No GPU required** — runs on CPU in ~2-3 minutes with `--quick` settings.

## 1. Install Dependencies

We need: `torch` (the engine), `numpy` (numerics), `tqdm` (progress bars), `matplotlib` (plots), and `ipywidgets` (interactive chat).

In [ ]:
%pip install torch numpy tqdm matplotlib ipywidgets -q
print('Dependencies ready.')

## 2. Import Tabula Rasa Package

The project uses a `src/` layout, so we add `src` to the Python path first.

In [ ]:
import sys, os, math, time, json
from pathlib import Path

# Add src/ so we can import tabula_rasa
notebook_dir = %pwd
sys.path.insert(0, os.path.join(notebook_dir, 'src'))

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch import optim
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from tabula_rasa.config import Config
from tabula_rasa.tokenizer import MathTokenizer
from tabula_rasa.model import MathTransformer, count_parameters

print(f'Torch {torch.__version__} | Tabula Rasa package imported OK')
print(f'CUDA available: {torch.cuda.is_available()}')

## 3. Configure and Initialize

We use `--quick` mode: **200 steps**, **2,000 training samples**, tiny evaluations every 100 steps. This runs in ~2-3 minutes on a laptop CPU.

In [ ]:
# ── Detect device ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Build tokenizer ──
tok = MathTokenizer()
print(f'Vocab size: {tok.vocab_size}')
print(f'Token IDs: PAD={tok.pad_id} BOS={tok.bos_id} EOS={tok.eos_id}')

# Test tokenizer
test_ids = tok.encode('12+34=0406', add_special_tokens=True)
print(f'Encode "12+34=0406": {test_ids}')
print(f'Decode back: {tok.decode(test_ids, skip_special=True)}')

# ── Config (quick mode tweaks) ──
cfg = Config()
cfg.vocab_size = tok.vocab_size
cfg.max_steps = 200           # Quick: 200 steps
cfg.train_samples = 2000      # Quick: 2K samples
cfg.batch_size = 64           # Quick: bigger batches for CPU
cfg.eval_every = 100          # Evaluate every 100 steps
cfg.eval_samples = 20         # Small eval set for speed
cfg.save_every = 999999       # Don't save checkpoints during notebook run
cfg.use_curriculum = False    # No curriculum in quick mode
cfg.min_digits = 1
cfg.max_digits = 4
cfg.use_reversed = True       # Reversed digits for add (aligns carry)
cfg.use_scratchpad = True     # Show carry scratchpad
cfg.use_loss_masking = True   # ignore_index=-100 for prompt
cfg.eval_temperature = 0.0    # Greedy eval
cfg.eval_max_tokens = 20
cfg.log_every = 25            # Log every 25 steps

print(f'\nConfig:')
print(f'  Model: d={cfg.d_model} L={cfg.n_layers} ff={cfg.d_ff} heads={cfg.n_heads}')
print(f'  Training: {cfg.max_steps} steps, {cfg.train_samples} samples, batch={cfg.batch_size}')
print(f'  Digits: {cfg.min_digits}-{cfg.max_digits}')
print(f'  Reversed: {cfg.use_reversed} | Scratchpad: {cfg.use_scratchpad} | Loss mask: {cfg.use_loss_masking}')

## 4. Generate Training Data

We build a dataset of addition problems with carry-digit scratchpad tokens.
Each sample is auto-padded to `max_seq_len` and loss-masked so the model only learns to predict the answer portion.

In [ ]:
# ── Problem generation (same logic as train_specialist.py) ──
OPS = {'add': '+', 'sub': '-', 'mul': '*', 'div': '/'}

def generate_problem(op: str, min_digits=1, max_digits=4):
    """Generate a single addition problem with scratchpad answer."""
    a_digits = random.randint(min_digits, max_digits)
    b_digits = random.randint(min_digits, max_digits)
    a = random.randint(10**(a_digits-1), 10**a_digits - 1)
    b = random.randint(10**(b_digits-1), 10**b_digits - 1)

    if op == 'add':
        ans = a + b
    elif op == 'sub':
        if a < b:
            a, b = b, a
        ans = a - b
    elif op == 'mul':
        ans = a * b
    elif op == 'div':
        ans = random.randint(1, max(1, a // 2))
        b = random.randint(1, max(1, a // 2))
        if b == 0: b = 1
        a = b * ans
        if a == 0:
            a = b * max(1, ans)
        ans = a // b if b != 0 else 1

    # Reversed digits + scratchpad for add/sub
    a_str = str(a)[::-1]
    b_str = str(b)[::-1]
    sp = ''
    carry = 0
    max_len = max(len(str(a)), len(str(b)))
    ra, rb = str(a)[::-1], str(b)[::-1]
    for i in range(max_len):
        da = int(ra[i]) if i < len(ra) else 0
        db = int(rb[i]) if i < len(rb) else 0
        total = da + db + carry
        carry = total // 10
        digit = total % 10
        sp += f'{carry}{digit}'
    ans_str = sp

    return f'{a_str}{OPS[op]}{b_str}', ans_str


import random

class SpecialistDataset(Dataset):
    """Dataset of addition problems for specialist training."""

    def __init__(self, tokenizer, op: str, cfg: Config):
        self.tokenizer = tokenizer
        self.max_seq_len = cfg.max_seq_len
        self.use_loss_masking = cfg.use_loss_masking
        self.pad_id = tokenizer.pad_id
        self.samples = []
        max_attempts = cfg.train_samples * 3
        for _ in range(max_attempts):
            if len(self.samples) >= cfg.train_samples:
                break
            expr, ans = generate_problem(op, cfg.min_digits, cfg.max_digits)
            ids = tokenizer.encode(f'{expr}={ans}', add_special_tokens=True)
            if len(ids) > self.max_seq_len:
                continue
            self.samples.append(ids)
        print(f'  Created {len(self.samples)} {op} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        padded = ids + [self.pad_id] * (self.max_seq_len - len(ids))
        x = torch.tensor(padded[:-1], dtype=torch.long)
        y = torch.tensor(padded[1:], dtype=torch.long)
        if self.use_loss_masking:
            y[y == self.pad_id] = -100
            eq_id = self.tokenizer.stoi['=']
            eq_positions = (y == eq_id).nonzero(as_tuple=True)[0]
            if len(eq_positions) > 0:
                eq_pos = eq_positions[0].item()
                y[:eq_pos + 1] = -100
        return x, y


t0 = time.time()
train_ds = SpecialistDataset(tok, 'add', cfg)
print(f'Dataset built in {time.time()-t0:.1f}s')
print(f'Sample: {tok.decode(train_ds.samples[0], skip_special=True)}')

# Show a few raw problems
print('\nSample problems:')
for i in range(5):
    expr, ans = generate_problem('add', 1, 2)
    print(f'  {expr} = {ans}')

## 5. Create the Model

A tiny transformer with **~1M parameters**: 4 layers, 128 hidden dim, 4 heads, 512 FF dimension. Uses **RoPE** position encoding and **SwiGLU** activation.

In [ ]:
model = MathTransformer(cfg).to(device)
n_params = count_parameters(model)
print(f'Model has {n_params:,} trainable parameters')
print(f'  d_model={cfg.d_model}, n_layers={cfg.n_layers}, n_heads={cfg.n_heads}, d_ff={cfg.d_ff}')
print(f'  pos_encoding={cfg.pos_encoding}, activation={cfg.activation}, norm={cfg.norm_type}')

# Quick sanity check: forward pass
sample_x, sample_y = train_ds[0]
logits, loss, value = model(sample_x.unsqueeze(0).to(device), sample_y.unsqueeze(0).to(device))
print(f'\nSanity forward pass:')
print(f'  Input:  {sample_x.tolist()}')
print(f'  Target: {sample_y.tolist()}')
print(f'  Logits shape: {logits.shape}')
print(f'  Loss (random init): {loss.item():.4f}  (should be ~ln({cfg.vocab_size})={math.log(cfg.vocab_size):.4f})')

## 6. Train the Specialist

This runs the core training loop from `train_specialist.py` directly in the notebook. We track:
- **Loss** at each log step
- **Accuracy** at each evaluation step
- **Per-digit accuracy** breakdown

On CPU this takes **~2-3 minutes** for 200 steps.

In [ ]:
# ── Helper: optimizer + LR schedule ──
def _make_optimizer(model, cfg):
    if cfg.optimizer == 'sgd':
        return optim.SGD(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    elif cfg.optimizer == 'adam':
        return optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay,
                          betas=(cfg.adam_beta1, cfg.adam_beta2))
    return optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay,
                       betas=(cfg.adam_beta1, cfg.adam_beta2))

def _get_lr(step, warmup, steps, schedule='cosine'):
    if schedule == 'constant':
        return 1.0
    if step < warmup:
        return step / max(1, warmup)
    p = (step - warmup) / max(1, steps - warmup)
    if schedule == 'linear':
        return max(0.01, 1.0 - p)
    return 0.5 * (1.0 + math.cos(math.pi * p))


# ── Helper: parse scratchpad answer ──
def parse_scratchpad_answer(text: str) -> str:
    text = text.replace('<EOS>', '').replace('<PAD>', '').replace('<BOS>', '').strip()
    if '=' in text:
        text = text.split('=')[-1]
    if len(text) % 2 != 0:
        text = text[:-1]
    return text[1::2]


# ── Helper: evaluate ──
@torch.no_grad()
def evaluate(model, tokenizer, cfg, op, num=0, min_digits=0, max_digits=0):
    model.eval()
    num = num or cfg.eval_samples
    min_d = min_digits or cfg.min_digits
    max_d = max_digits or cfg.max_digits
    correct = 0
    for _ in range(num):
        expr, ans_raw = generate_problem(op, min_d, max_d)
        prompt = f'{expr}='
        out = model.generate(tokenizer, prompt,
                             max_new_tokens=cfg.eval_max_tokens,
                             temperature=cfg.eval_temperature, top_k=3)
        pred = parse_scratchpad_answer(out)
        true_answer = parse_scratchpad_answer(ans_raw)
        if pred == true_answer:
            correct += 1
    return correct / max(1, num) * 100


@torch.no_grad()
def evaluate_per_digit(model, tokenizer, cfg, op, per_digit_samples=25):
    model.eval()
    results = {}
    for d in range(cfg.min_digits, cfg.max_digits + 1):
        correct = 0
        for _ in range(per_digit_samples):
            expr, ans_raw = generate_problem(op, d, d)
            prompt = f'{expr}='
            out = model.generate(tokenizer, prompt,
                                 max_new_tokens=cfg.eval_max_tokens,
                                 temperature=cfg.eval_temperature, top_k=3)
            pred = parse_scratchpad_answer(out)
            true_answer = parse_scratchpad_answer(ans_raw)
            if pred == true_answer:
                correct += 1
        results[d] = correct / max(1, per_digit_samples) * 100
    return results


# ── TRAINING LOOP ──
print(f'\n{"="*60}')
print(f'  Training Addition Specialist')
print(f'{"="*60}')

optimizer = _make_optimizer(model, cfg)
global_step = 0
best_acc = 0.0

# Tracking for plots
step_history = []
loss_history = []
eval_step_history = []
eval_acc_history = []
per_digit_history = []

t_start = time.time()
print(f'  Training {cfg.max_steps} steps...\n')

pbar = tqdm(total=cfg.max_steps, desc='Training', unit='step')

while global_step < cfg.max_steps:
    loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                        num_workers=0, drop_last=True)
    optimizer.zero_grad()

    for batch in loader:
        if global_step >= cfg.max_steps:
            break

        x, y = batch
        x, y = x.to(device), y.to(device)

        _, loss, _ = model(x, y)
        loss.backward()

        if cfg.grad_clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)

        optimizer.step()
        optimizer.zero_grad()
        global_step += 1

        # Learning rate schedule
        for pg in optimizer.param_groups:
            pg['lr'] = cfg.learning_rate * _get_lr(
                global_step, cfg.warmup_steps, cfg.max_steps, cfg.lr_schedule)

        # Log step
        if global_step % cfg.log_every == 0:
            step_history.append(global_step)
            loss_history.append(loss.item())
            current_lr = cfg.learning_rate * _get_lr(global_step, cfg.warmup_steps, cfg.max_steps, cfg.lr_schedule)
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{current_lr:.6f}'})

        pbar.update(1)

        # Evaluation
        if global_step % cfg.eval_every == 0:
            acc = evaluate(model, tok, cfg, 'add')
            per_digit = evaluate_per_digit(model, tok, cfg, 'add', per_digit_samples=15)

            eval_step_history.append(global_step)
            eval_acc_history.append(acc)
            per_digit_history.append(per_digit)

            digit_str = ' '.join(f'{d}d:{v:.0f}%' for d, v in per_digit.items())
            tqdm.write(
                f'  Eval step {global_step:>4}: {acc:.1f}% (best: {max(best_acc, acc):.1f}%) | {digit_str}')

            if acc > best_acc:
                best_acc = acc

pbar.close()

elapsed = time.time() - t_start
print(f'\n  Done! {elapsed:.0f}s ({elapsed/60:.1f} min) | Best: {best_acc:.1f}% | Steps: {global_step}')
print(f'{"="*60}')

## 7. Visualize Training

Plot the **loss curve** and **accuracy progression** over the training run.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1 = axes[0]
ax1.plot(step_history, loss_history, 'b-', linewidth=2)
ax1.set_xlabel('Training Step')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)
if len(loss_history) > 1:
    # Add smoothed overlay
    window = max(1, len(loss_history) // 10)
    smoothed = np.convolve(loss_history, np.ones(window)/window, mode='valid')
    ax1.plot(step_history[:len(smoothed)], smoothed, 'r--', linewidth=1.5, alpha=0.7, label='Smoothed')
    ax1.legend()

# Accuracy progression
ax2 = axes[1]
if eval_step_history:
    ax2.plot(eval_step_history, eval_acc_history, 'g-', marker='o', linewidth=2)
    ax2.set_xlabel('Training Step')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Overall Accuracy')
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(-5, 105)

plt.tight_layout()
plt.show()

print(f'Final accuracy: {eval_acc_history[-1]:.1f}%' if eval_acc_history else 'No eval performed.')

## 8. Per-Digit Accuracy Breakdown

See how well the model performs on 1-digit, 2-digit, 3-digit, and 4-digit additions.

In [ ]:
# Final per-digit accuracy
print('Per-Digit Accuracy (final):')
if per_digit_history:
    final_per_digit = per_digit_history[-1]
    digits = list(final_per_digit.keys())
    accs = [final_per_digit[d] for d in digits]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar([f'{d}-digit' for d in digits], accs, color=['#4CAF50', '#2196F3', '#FF9800', '#F44336'][:len(digits)])
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Accuracy by Digit Length')
    ax.set_ylim(0, 105)
    ax.grid(True, axis='y', alpha=0.3)

    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{acc:.0f}%', ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Also show progression over training
    if len(per_digit_history) > 1:
        print('\nPer-digit accuracy progression:')
        header = f'{"Step":>6}' + ''.join(f'{"  {d}d":>8}' for d in digits)
        print(header)
        print('-' * len(header))
        for i, (step, pd) in enumerate(zip(eval_step_history, per_digit_history)):
            row = f'{step:>6}' + ''.join(f'{pd.get(d, 0):>8.0f}' for d in digits)
            print(row)
else:
    print('No evaluation data available.')

## 9. Interactive Chat

Type an addition problem (e.g. `15+27`) and see the model's answer. The model uses reversed digits + scratchpad internally, but we convert its output to a readable answer.

**Note:** This model was trained with **quick mode** (200 steps, 2K samples), so it won't be perfectly accurate — especially on larger numbers. Train with full settings (`python3 train_specialist.py add`) for production quality.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def solve_problem(expr: str) -> str:
    """Ask the model to solve an addition problem."""
    if '+' not in expr:
        return "Please enter an addition problem like '15+27'"
    
    prompt = f'{expr}='
    out = model.generate(tok, prompt,
                         max_new_tokens=cfg.eval_max_tokens,
                         temperature=0.0, top_k=3)
    
    # Parse scratchpad output
    raw_pred = out.replace('<EOS>', '').replace('<PAD>', '').replace('<BOS>', '').strip()
    if '=' in raw_pred:
        sp_part = raw_pred.split('=')[-1]
        sp_part = sp_part.strip()
        # Extract digits from scratchpad (every second char = actual digit)
        if len(sp_part) % 2 == 0:
            digits = sp_part[1::2]  # LSD-first
            answer = digits[::-1]  # Reverse for human reading
            return answer
    
    # Fallback: just show raw output
    return f"raw: {repr(raw_pred)}"


# ── Widget UI ──
input_box = widgets.Text(
    value='15+27',
    placeholder='Enter addition (e.g. 15+27)',
    description='Problem:',
    layout=widgets.Layout(width='400px')
)
solve_btn = widgets.Button(description='Solve', button_style='primary')
output_area = widgets.Output()

def on_solve_clicked(b):
    output_area.clear_output()
    with output_area:
        expr = input_box.value.strip()
        if not expr:
            print('Please enter a problem.')
            return
        print(f'Problem:  {expr}')
        print(f'Answer:   {solve_problem(expr)}')

solve_btn.on_click(on_solve_clicked)

display(widgets.VBox([
    widgets.HTML('<b>Interactive Math Solver</b>'),
    widgets.HBox([input_box, solve_btn]),
    output_area
]))

## 10. Quick Batch Test

Test the model on a set of curated problems and see pass/fail results.

In [ ]:
print(f'{"Problem":<20} {"Model Answer":<20} {"Expected":<12} {"Result":<8}')
print('-' * 60)

test_problems = [
    ('3+4=', '07'),
    ('7+8=', '15'),
    ('12+34=', '46'),
    ('56+17=', '73'),
    ('99+1=', '100'),
    ('123+456=', '579'),
    ('500+500=', '1000'),
]

correct = 0
total = len(test_problems)

for expr, expected_scratchpad in test_problems:
    out = model.generate(tok, expr, max_new_tokens=cfg.eval_max_tokens,
                         temperature=0.0, top_k=3)
    pred = parse_scratchpad_answer(out)
    result = 'PASS' if pred == expected_scratchpad else 'FAIL'
    if result == 'PASS':
        correct += 1
    print(f'{expr:<20} {pred:<20} {expected_scratchpad:<12} {result:<8}')

print(f'\nBatch accuracy: {correct}/{total} = {correct/total*100:.0f}%')

## Next Steps

This quickstart trained a tiny model for just 200 steps. For real results:

- **Full training**: `python3 train_specialist.py add` (30K steps, takes ~1-2 hours)
- **Other operations**: `train_specialist.py sub`, `mul`, `div`
- **Continual learning**: `train_specialist.py sub --ewc` (preserves addition while learning subtraction)
- **Deeper model**: `train_specialist.py add --deep` (d=64 L=8, better for algorithms)
- **API server**: `python3 api_server.py` to serve the model via REST API

See the [README](https://github.com/NousResearch/tabula-rasa) and [CONTRIBUTING.md](CONTRIBUTING.md) for more details.